# VOC Ensemble and Evaluation Notebook

Combines predictions from DETR, YOLO, Faster R-CNN, and NCA using weighted box fusion-like averaging after class-wise NMS.

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
PROJECT_ROOT = Path.cwd()
TRAINVAL_ROOT = PROJECT_ROOT / 'trainval' / 'VOC2012'
TEST_ROOT = PROJECT_ROOT / 'test' / 'VOC2012'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
SPLIT_DIR = ARTIFACTS_DIR / 'group_split'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}

PREDICTION_COLUMNS = [
    'image_id', 'split', 'source_model', 'class_id', 'cls',
    'xmin', 'ymin', 'xmax', 'ymax', 'score'
]

for path in [TRAINVAL_ROOT, TEST_ROOT]:
    if not path.exists():
        raise FileNotFoundError(f'Missing expected VOC folder: {path}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SPLIT_DIR:', SPLIT_DIR)

In [ ]:
def read_ids_from_file(path: Path) -> list[str]:
    if not path.exists():
        return []
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]


def load_split_ids(split_name: str) -> list[str]:
    # Prefer locked team split files to avoid leakage.
    shared_path = SPLIT_DIR / f'{split_name}_ids.txt'
    shared_ids = read_ids_from_file(shared_path)
    if shared_ids:
        return shared_ids

    # Fallback to VOC default lists if shared split files are missing.
    root = TRAINVAL_ROOT if split_name in {'train', 'val'} else TEST_ROOT
    fallback_path = root / 'ImageSets' / 'Main' / f'{split_name}.txt'
    return read_ids_from_file(fallback_path)


def parse_annotation(xml_path: Path, image_id: str, split_name: str) -> tuple[dict, list[dict]]:
    root = ET.parse(xml_path).getroot()

    size_node = root.find('size')
    width = int(size_node.findtext('width', default='0')) if size_node is not None else 0
    height = int(size_node.findtext('height', default='0')) if size_node is not None else 0

    records = []
    for obj in root.findall('object'):
        cls = obj.findtext('name', default='').strip()
        if cls not in CLASS_TO_ID:
            continue

        box = obj.find('bndbox')
        if box is None:
            continue

        xmin = int(float(box.findtext('xmin', default='0')))
        ymin = int(float(box.findtext('ymin', default='0')))
        xmax = int(float(box.findtext('xmax', default='0')))
        ymax = int(float(box.findtext('ymax', default='0')))

        box_w = max(1, xmax - xmin + 1)
        box_h = max(1, ymax - ymin + 1)

        records.append({
            'image_id': image_id,
            'split': split_name,
            'cls': cls,
            'class_id': CLASS_TO_ID[cls],
            'xmin': xmin,
            'ymin': ymin,
            'xmax': xmax,
            'ymax': ymax,
            'box_w': box_w,
            'box_h': box_h,
            'box_area': box_w * box_h,
            'difficult': int(obj.findtext('difficult', default='0')),
            'truncated': int(obj.findtext('truncated', default='0')),
            'occluded': int(obj.findtext('occluded', default='0')),
            'width': width,
            'height': height,
        })

    image_meta = {
        'image_id': image_id,
        'split': split_name,
        'width': width,
        'height': height,
    }
    return image_meta, records


def build_split_tables(root_dir: Path, split_name: str, image_ids: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    image_rows, object_rows = [], []
    ann_dir = root_dir / 'Annotations'

    for image_id in image_ids:
        xml_path = ann_dir / f'{image_id}.xml'
        if not xml_path.exists():
            image_rows.append({'image_id': image_id, 'split': split_name, 'width': np.nan, 'height': np.nan})
            continue

        image_meta, rows = parse_annotation(xml_path, image_id, split_name)
        image_rows.append(image_meta)
        object_rows.extend(rows)

    images_df = pd.DataFrame(image_rows).drop_duplicates(subset=['image_id', 'split'])
    objects_df = pd.DataFrame(object_rows)
    return images_df, objects_df


train_ids = load_split_ids('train')
val_ids = load_split_ids('val')
test_ids = load_split_ids('test')

if not train_ids or not val_ids or not test_ids:
    raise RuntimeError('Missing one or more split files. Check artifacts/group_split or ImageSets/Main.')

train_images, train_objects = build_split_tables(TRAINVAL_ROOT, 'train', train_ids)
val_images, val_objects = build_split_tables(TRAINVAL_ROOT, 'val', val_ids)
test_images, test_objects = build_split_tables(TEST_ROOT, 'test', test_ids)

train_model_df = train_objects[(train_objects['xmax'] >= train_objects['xmin']) & (train_objects['ymax'] >= train_objects['ymin'])].copy()
val_model_df = val_objects[(val_objects['xmax'] >= val_objects['xmin']) & (val_objects['ymax'] >= val_objects['ymin'])].copy()

train_model_df_ignoring_difficult = train_model_df[train_model_df['difficult'] == 0].reset_index(drop=True)
val_model_df_ignoring_difficult = val_model_df[val_model_df['difficult'] == 0].reset_index(drop=True)

print('Split counts:', len(train_ids), len(val_ids), len(test_ids))
print('train-val overlap:', len(set(train_ids) & set(val_ids)))
print('train-test overlap:', len(set(train_ids) & set(test_ids)))
print('val-test overlap:', len(set(val_ids) & set(test_ids)))

display(train_model_df_ignoring_difficult.head())

In [ ]:
def compute_iou_xyxy(box_a: np.ndarray, box_b: np.ndarray) -> float:
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0.0, inter_x2 - inter_x1 + 1)
    inter_h = max(0.0, inter_y2 - inter_y1 + 1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, xa2 - xa1 + 1) * max(0.0, ya2 - ya1 + 1)
    area_b = max(0.0, xb2 - xb1 + 1) * max(0.0, yb2 - yb1 + 1)

    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union


def ap_from_pr(precisions: np.ndarray, recalls: np.ndarray) -> float:
    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def evaluate_map50(pred_df: pd.DataFrame, gt_df: pd.DataFrame, classes: list[str], iou_thresh: float = 0.5) -> dict:
    if pred_df.empty:
        return {'map50': 0.0, 'per_class_ap': {cls: 0.0 for cls in classes}}

    per_class_ap = {}
    for cls in classes:
        gt_cls = gt_df[gt_df['cls'] == cls]
        pred_cls = pred_df[pred_df['cls'] == cls].sort_values('score', ascending=False)

        gt_by_image = {}
        for image_id, group in gt_cls.groupby('image_id'):
            gt_by_image[image_id] = {
                'boxes': group[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float),
                'matched': np.zeros(len(group), dtype=bool),
            }

        tps = np.zeros(len(pred_cls), dtype=float)
        fps = np.zeros(len(pred_cls), dtype=float)

        for i, row in enumerate(pred_cls.itertuples(index=False)):
            image_state = gt_by_image.get(row.image_id)
            if image_state is None or len(image_state['boxes']) == 0:
                fps[i] = 1.0
                continue

            pred_box = np.array([row.xmin, row.ymin, row.xmax, row.ymax], dtype=float)
            ious = np.array([compute_iou_xyxy(pred_box, gt_box) for gt_box in image_state['boxes']])
            best_idx = int(np.argmax(ious))
            best_iou = ious[best_idx]

            if best_iou >= iou_thresh and not image_state['matched'][best_idx]:
                tps[i] = 1.0
                image_state['matched'][best_idx] = True
            else:
                fps[i] = 1.0

        cum_tp = np.cumsum(tps)
        cum_fp = np.cumsum(fps)
        recalls = cum_tp / max(1, len(gt_cls))
        precisions = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
        per_class_ap[cls] = ap_from_pr(precisions, recalls) if len(pred_cls) > 0 else 0.0

    map50 = float(np.mean(list(per_class_ap.values()))) if per_class_ap else 0.0
    return {'map50': map50, 'per_class_ap': per_class_ap}


def ensure_prediction_schema(df: pd.DataFrame, source_model: str, split_name: str) -> pd.DataFrame:
    out = df.copy()
    out['source_model'] = source_model
    out['split'] = split_name
    for col in PREDICTION_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
    out = out[PREDICTION_COLUMNS]
    out['class_id'] = out['class_id'].astype('Int64')
    return out

In [ ]:
MODEL_FILES = {
    'detr': PREDICTIONS_DIR / 'detr_val_predictions.csv',
    'yolo': PREDICTIONS_DIR / 'yolo_val_predictions.csv',
    'fasterrcnn': PREDICTIONS_DIR / 'fasterrcnn_val_predictions.csv',
    'nca': PREDICTIONS_DIR / 'nca_val_predictions.csv',
}

MODEL_WEIGHTS = {
    'detr': 1.0,
    'yolo': 1.0,
    'fasterrcnn': 1.0,
    'nca': 1.0,
}


def nms_classwise(df: pd.DataFrame, iou_thresh: float = 0.55) -> pd.DataFrame:
    rows = []
    for (_, _), group in df.groupby(['image_id', 'class_id']):
        g = group.sort_values('score', ascending=False).reset_index(drop=True)
        keep = []
        suppressed = np.zeros(len(g), dtype=bool)

        for i in range(len(g)):
            if suppressed[i]:
                continue
            keep.append(i)
            box_i = g.loc[i, ['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float)
            for j in range(i + 1, len(g)):
                if suppressed[j]:
                    continue
                box_j = g.loc[j, ['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float)
                if compute_iou_xyxy(box_i, box_j) >= iou_thresh:
                    suppressed[j] = True

        rows.append(g.loc[keep])

    if not rows:
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    return pd.concat(rows, ignore_index=True)


def fuse_group(group: pd.DataFrame) -> dict:
    weighted = []
    for row in group.itertuples(index=False):
        model_w = MODEL_WEIGHTS.get(row.source_model, 1.0)
        weighted.append(float(row.score) * model_w)
    weighted = np.asarray(weighted, dtype=float)
    total = max(weighted.sum(), 1e-12)

    xmin = float(np.sum(group['xmin'].to_numpy(dtype=float) * weighted) / total)
    ymin = float(np.sum(group['ymin'].to_numpy(dtype=float) * weighted) / total)
    xmax = float(np.sum(group['xmax'].to_numpy(dtype=float) * weighted) / total)
    ymax = float(np.sum(group['ymax'].to_numpy(dtype=float) * weighted) / total)

    return {
        'image_id': group.iloc[0]['image_id'],
        'split': group.iloc[0]['split'],
        'source_model': 'ensemble',
        'class_id': int(group.iloc[0]['class_id']),
        'cls': group.iloc[0]['cls'],
        'xmin': xmin,
        'ymin': ymin,
        'xmax': xmax,
        'ymax': ymax,
        'score': float(group['score'].max()),
    }


def build_ensemble_predictions(pred_df: pd.DataFrame, iou_thresh: float = 0.55) -> pd.DataFrame:
    fused_rows = []
    for (_, _), group in pred_df.groupby(['image_id', 'class_id']):
        g = group.sort_values('score', ascending=False).reset_index(drop=True)
        used = np.zeros(len(g), dtype=bool)

        for i in range(len(g)):
            if used[i]:
                continue
            box_i = g.loc[i, ['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float)
            cluster = [i]
            used[i] = True
            for j in range(i + 1, len(g)):
                if used[j]:
                    continue
                box_j = g.loc[j, ['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float)
                if compute_iou_xyxy(box_i, box_j) >= iou_thresh:
                    cluster.append(j)
                    used[j] = True

            fused_rows.append(fuse_group(g.loc[cluster]))

    out = pd.DataFrame(fused_rows)
    if out.empty:
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    return out[PREDICTION_COLUMNS]

In [ ]:
val_parts = []
missing = []
for _, path in MODEL_FILES.items():
    if path.exists():
        df = pd.read_csv(path)
        if not df.empty:
            val_parts.append(df)
    else:
        missing.append(path.name)

if missing:
    print('Missing prediction files:', missing)

if not val_parts:
    raise RuntimeError('No validation prediction files found in artifacts/predictions.')

val_all = pd.concat(val_parts, ignore_index=True)
val_all = val_all[val_all['split'] == 'val'].copy()

val_all = nms_classwise(val_all, iou_thresh=0.55)
val_ens = build_ensemble_predictions(val_all, iou_thresh=0.55)

metrics_ens = evaluate_map50(val_ens, val_model_df_ignoring_difficult, VOC_CLASSES)
print('Ensemble val mAP@0.5:', round(metrics_ens['map50'], 4))

display(val_ens.head())

In [ ]:
TEST_MODEL_FILES = {
    'detr': PREDICTIONS_DIR / 'detr_test_predictions.csv',
    'yolo': PREDICTIONS_DIR / 'yolo_test_predictions.csv',
    'fasterrcnn': PREDICTIONS_DIR / 'fasterrcnn_test_predictions.csv',
    'nca': PREDICTIONS_DIR / 'nca_test_predictions.csv',
}

test_parts = []
for _, path in TEST_MODEL_FILES.items():
    if path.exists():
        df = pd.read_csv(path)
        if not df.empty:
            test_parts.append(df)

if test_parts:
    test_all = pd.concat(test_parts, ignore_index=True)
    test_all = test_all[test_all['split'] == 'test'].copy()
    test_all = nms_classwise(test_all, iou_thresh=0.55)
    test_ens = build_ensemble_predictions(test_all, iou_thresh=0.55)
else:
    test_ens = pd.DataFrame(columns=PREDICTION_COLUMNS)

out_path = PREDICTIONS_DIR / 'ensemble_test_predictions.csv'
test_ens.to_csv(out_path, index=False)
print('Saved test ensemble predictions to:', out_path)
print('Rows:', len(test_ens))